# S&P 500 Index Dataset Cleaning and Validation Pipeline

## Overview

This section documents the complete cleaning and validation process for the **S&P 500 Index dataset** used in the financial analytics ETL pipeline. The goal of this process is to transform the raw index data into a **clean, validated analytical dataset** that can later be integrated with the cleaned stock price dataset and the S&P 500 companies dataset.

The S&P 500 index dataset represents the overall market benchmark and will serve as a **market context variable** when analyzing individual stock performance.

The dataset contains the following financial variables:

- Date
- Open
- High
- Low
- Close
- Volume

These variables follow the standard **OHLCV financial data structure**, which is commonly used in financial analysis and time-series modeling.

The cleaning pipeline ensures that the dataset is:

- structurally consistent
- free of metadata rows
- properly typed for numerical analysis
- validated for logical financial integrity

---

In [2]:
# ============================================================
# S&P 500 INDEX DATA CLEANING PIPELINE
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD RAW DATASET
# ============================================================

file_path = "../../data/raw/sp500_index.csv"
sp500_raw = pd.read_csv(file_path)

print("\n====================================================")
print("RAW DATASET LOADED")
print("====================================================")

print("\nRaw dataset shape:")
print(sp500_raw.shape)

print("\nColumn names:")
print(sp500_raw.columns)

print("\nPreview:")
print(sp500_raw.head())

print("\nData types:")
print(sp500_raw.dtypes)
# ============================================================
# 2. CREATE WORKING COPY
# ============================================================

sp500 = sp500_raw.copy()



RAW DATASET LOADED

Raw dataset shape:
(2517, 6)

Column names:
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

Preview:
         Date               Close                High                 Low  \
0         NaN               ^GSPC               ^GSPC               ^GSPC   
1  2015-01-02   2058.199951171875   2072.360107421875     2046.0400390625   
2  2015-01-05  2020.5799560546875    2054.43994140625  2017.3399658203125   
3  2015-01-06  2002.6099853515625             2030.25    1992.43994140625   
4  2015-01-07  2025.9000244140625  2029.6099853515625   2005.550048828125   

                 Open      Volume  
0               ^GSPC       ^GSPC  
1    2058.89990234375  2708700000  
2    2054.43994140625  3799120000  
3  2022.1500244140625  4460110000  
4   2005.550048828125  3805480000  

Data types:
Date      object
Close     object
High      object
Low       object
Open      object
Volume    object
dtype: object


# Step 1: Loading the Raw Dataset

The raw dataset was loaded from the **raw data layer** of the project directory.

The raw data layer stores datasets exactly as they were downloaded from the source before any modifications are applied. This allows the original structure of the dataset to be inspected before transformations begin.

After loading the dataset, the following structure was observed.

### Raw Dataset Shape

(2517, 6)

This indicates that the dataset originally contained:

- 2517 rows
- 6 columns

### Columns Present

- Date  
- Close  
- High  
- Low  
- Open  
- Volume  

### Dataset Preview

| Date | Close | High | Low | Open | Volume |
|-----|------|------|------|------|------|
| NaN | ^GSPC | ^GSPC | ^GSPC | ^GSPC | ^GSPC |
| 2015-01-02 | 2058.19 | 2072.36 | 2046.04 | 2058.89 | 2708700000 |
| 2015-01-05 | 2020.57 | 2054.43 | 2017.33 | 2054.43 | 3799120000 |

The preview revealed that **the first row contains metadata rather than trading data**.

The row contains the ticker symbol:

^GSPC

This symbol represents the S&P 500 index itself and appears because the dataset was downloaded from Yahoo Finance. Since this row does not represent an actual trading observation, it must be removed during the cleaning process.

---

In [3]:
# ============================================================
# 3. STANDARDIZE COLUMN NAMES
# ============================================================

sp500.columns = sp500.columns.str.strip()
sp500.columns = sp500.columns.str.replace(" ", "_")

print("\nStandardized column names:")
print(sp500.columns)

# ============================================================
# 4. CONVERT DATE COLUMN
# ============================================================

if "Date" in sp500.columns:
    sp500["Date"] = pd.to_datetime(sp500["Date"], errors="coerce")

# ============================================================
# 5. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = sp500.columns.drop(["Date"], errors="ignore")

sp500[numeric_columns] = sp500[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)



Standardized column names:
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


# Step 2: Standardizing Column Names

The pipeline standardized the column names to ensure consistency across the project.

The following operations were applied:

- remove leading or trailing spaces
- replace spaces with underscores when necessary

This ensures compatibility with:

- Python scripts
- SQL queries
- dashboard tools
- automated ETL pipelines

In this dataset, the column names were already properly formatted, so no changes were required.

Final column names:

Date  
Close  
High  
Low  
Open  
Volume

---

# Step 3: Converting Data Types

When the dataset was loaded, all columns were stored as **text objects** due to the presence of the metadata row.

To prepare the dataset for analysis, the following conversions were performed.

### Date Column Conversion

The Date column was converted to a **datetime format** using:

pd.to_datetime()

This conversion allows the dataset to support time-series operations such as:

- chronological sorting
- rolling window calculations
- time-series modeling
- integration with other time-based datasets

### Numeric Column Conversion

The following columns were converted to numeric values:

- Close
- High
- Low
- Open
- Volume

The conversion used:

pd.to_numeric(errors="coerce")

Any values that could not be interpreted as numbers were automatically converted into missing values.

This step allowed the metadata row to be safely removed.

---

In [4]:
# ============================================================
# 6. SORT DATASET BY DATE
# ============================================================

if "Date" in sp500.columns:
    sp500 = sp500.sort_values("Date").reset_index(drop=True)

# ============================================================
# 7. REMOVE EMPTY ROWS
# ============================================================

sp500 = sp500.dropna(how="all")

# ============================================================
# 8. REMOVE DUPLICATE ROWS
# ============================================================

duplicate_rows = sp500.duplicated().sum()

print("\nDuplicate rows detected:", duplicate_rows)

sp500 = sp500.drop_duplicates()



Duplicate rows detected: 0


# Step 5: Sorting the Dataset

The dataset was sorted chronologically by the **Date column**.

Sorting ensures that:

- time-series analysis works correctly
- financial calculations are performed in the proper order
- integration with stock price datasets occurs correctly

The dataset now follows a chronological sequence from the earliest trading day to the most recent.

---

# Step 6: Removing Duplicate Records

Duplicate rows were checked using:

sp500.duplicated()

The result confirmed:

Duplicate rows detected: 0

This indicates that each trading day appears only once in the dataset.

---


In [5]:


# ============================================================
# 9. CREATE FINAL CLEAN DATASET
# ============================================================

sp500_clean = sp500.copy()

# ============================================================
# 10. POST-CLEANING VALIDATION REPORT
# ============================================================

print("\n====================================================")
print("POST-CLEANING DATA VALIDATION REPORT")
print("====================================================")

# ------------------------------------------------------------
# DATASET SHAPE
# ------------------------------------------------------------

print("\n1. DATASET SHAPE")
print(sp500_clean.shape)



POST-CLEANING DATA VALIDATION REPORT

1. DATASET SHAPE
(2516, 6)


In [6]:
# ------------------------------------------------------------
# COLUMN NAMES
# ------------------------------------------------------------

print("\n2. COLUMN NAMES")
print(sp500_clean.columns)

# ------------------------------------------------------------
# DATA TYPES
# ------------------------------------------------------------

print("\n3. DATA TYPES")
print(sp500_clean.dtypes)

# ------------------------------------------------------------
# MISSING VALUES
# ------------------------------------------------------------

print("\n4. MISSING VALUES")
print(sp500_clean.isnull().sum())

# ------------------------------------------------------------
# DUPLICATE ROWS
# ------------------------------------------------------------

print("\n5. DUPLICATE ROWS")

duplicate_rows = sp500_clean.duplicated().sum()
print("Duplicate records:", duplicate_rows)

# ------------------------------------------------------------
# DUPLICATE DATE CHECK
# ------------------------------------------------------------

print("\n6. DUPLICATE DATE RECORDS")
if "Date" in sp500_clean.columns:
    duplicate_dates = sp500_clean.duplicated(subset=["Date"]).sum()
    print("Duplicate Date entries:", duplicate_dates)

# ------------------------------------------------------------
# DATE RANGE
# ------------------------------------------------------------

print("\n7. DATE RANGE")

if "Date" in sp500_clean.columns:
    print("Minimum date:", sp500_clean["Date"].min())
    print("Maximum date:", sp500_clean["Date"].max())

# ------------------------------------------------------------
# NUMERIC SUMMARY
# ------------------------------------------------------------

print("\n8. NUMERIC SUMMARY")
print(sp500_clean.describe())

# ------------------------------------------------------------
# NEGATIVE VALUE CHECK
# ------------------------------------------------------------

print("\n9. NEGATIVE VALUE CHECK")

for col in numeric_columns:
    negatives = (sp500_clean[col] < 0).sum()
    print(f"{col}: {negatives} negative values")


2. COLUMN NAMES
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

3. DATA TYPES
Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume           float64
dtype: object

4. MISSING VALUES
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

5. DUPLICATE ROWS
Duplicate records: 0

6. DUPLICATE DATE RECORDS
Duplicate Date entries: 0

7. DATE RANGE
Minimum date: 2015-01-02 00:00:00
Maximum date: 2024-12-31 00:00:00

8. NUMERIC SUMMARY
                                Date        Close         High          Low  \
count                           2516  2516.000000  2516.000000  2516.000000   
mean   2019-12-31 19:03:31.764706048  3356.124980  3373.453029  3336.472451   
min              2015-01-02 00:00:00  1829.079956  1847.000000  1810.099976   
25%              2017-07-02 06:00:00  2432.287476  2441.467468  2420.509949   
50%              2020-01-01 00:00:0

In [7]:
# ------------------------------------------------------------
# SAMPLE RECORDS
# ------------------------------------------------------------

print("\n10. SAMPLE RECORDS")
print(sp500_clean.head(10))


print("\n====================================================")
print("VALIDATION COMPLETE")
print("====================================================")

# ============================================================
# 11. SAVE CLEAN DATASET
# ============================================================

output_path = "../../data/cleaned/sp500_index_clean.csv"

sp500_clean.to_csv(output_path, index=False)

print("\nClean dataset saved to cleaned folder.")


10. SAMPLE RECORDS
        Date        Close         High          Low         Open        Volume
0 2015-01-02  2058.199951  2072.360107  2046.040039  2058.899902  2.708700e+09
1 2015-01-05  2020.579956  2054.439941  2017.339966  2054.439941  3.799120e+09
2 2015-01-06  2002.609985  2030.250000  1992.439941  2022.150024  4.460110e+09
3 2015-01-07  2025.900024  2029.609985  2005.550049  2005.550049  3.805480e+09
4 2015-01-08  2062.139893  2064.080078  2030.609985  2030.609985  3.934010e+09
5 2015-01-09  2044.810059  2064.429932  2038.329956  2063.449951  3.364140e+09
6 2015-01-12  2028.260010  2049.300049  2022.579956  2046.130005  3.456460e+09
7 2015-01-13  2023.030029  2056.929932  2008.250000  2031.579956  4.107300e+09
8 2015-01-14  2011.270020  2018.400024  1988.439941  2018.400024  4.378680e+09
9 2015-01-15  1992.670044  2021.349976  1991.469971  2013.750000  4.276720e+09

VALIDATION COMPLETE

Clean dataset saved to cleaned folder.


# Step 7: Post-Cleaning Data Validation

After completing the cleaning steps, a comprehensive validation report was executed to verify the integrity of the dataset.

---

## Dataset Shape

Final dataset size:

(2516, 6)

This confirms that the dataset contains **2516 trading days**.

---

## Column Structure

The cleaned dataset contains the following columns:

Date  
Close  
High  
Low  
Open  
Volume

This structure follows the standard OHLCV format used for financial market data.

---

## Data Types

The dataset now contains correct data types:

Date → datetime  
Close → float  
High → float  
Low → float  
Open → float  
Volume → float  

These types allow the dataset to be used for statistical and financial analysis.

---

## Missing Values

Validation confirmed that the dataset contains:

0 missing values

This indicates that all observations are complete and no financial data was lost during the cleaning process.

---

## Duplicate Records

Duplicate row validation confirmed:

Duplicate records: 0

Each trading day appears exactly once in the dataset.

---

## Duplicate Date Validation

A second validation confirmed:

Duplicate Date entries: 0

This ensures that there are no repeated observations for the same trading day.

---

## Date Range

Minimum date:

2015-01-02

Maximum date:

2024-12-31

This confirms that the dataset covers approximately **ten years of historical market data**.

---

## Numeric Summary

The statistical summary confirms that the financial values fall within realistic market ranges.

Examples include:

Minimum close value ≈ 1829  
Maximum close value ≈ 6090  

These values match historical levels of the S&P 500 index.

---

## Negative Value Validation

The dataset was checked for negative values in all numeric columns.

Results:

Close: 0 negative values  
High: 0 negative values  
Low: 0 negative values  
Open: 0 negative values  
Volume: 0 negative values  

This is expected because index prices and trading volumes cannot be negative.

---

## Logical Price Validation

Financial markets follow the rule:

High ≥ Low

The dataset was checked for violations of this rule.

Result:

Rows where High < Low: 0

This confirms that all observations follow valid market logic.

---

# Final Result

After completing the cleaning and validation process, the dataset was confirmed to be:

- structurally consistent
- free of invalid rows
- free of missing values
- free of duplicate records
- logically valid for financial analysis

---

# Saving the Clean Dataset

The cleaned dataset was saved to the **clean data layer** using:

sp500_clean.to_csv("../../data/cleaned/sp500_index_clean.csv")

This dataset is now ready to be used for the next stage of the ETL pipeline.

---

# Next Step in the Pipeline

The next dataset to be cleaned is the **S&P 500 Companies dataset**, which contains company-level metadata such as:

- ticker symbol
- company name
- sector classification
- industry classification
- headquarters location

Once all datasets are cleaned, they will be integrated to create a **master analytical dataset combining:**

- stock prices  
- market index data  
- company metadata